# Ứng dụng 2: Mẫu trực quan trên ảnh (Caltech-101)

**Câu hỏi:** mỗi loại vật thể có những tổ hợp chi tiết hình ảnh nào đặc trưng, và các tổ hợp đó có giúp phân loại ảnh không?

Notebook này chỉ **điều phối**: mỗi bước là một file trong `applications/app2_image/src/`, ghi kết quả trung gian ra `applications/app2_image/data/` và `applications/app2_image/out/`.
Nếu Colab ngắt giữa chừng, chạy lại cell thiết lập rồi chạy tiếp từ bước đang dở (các bước bỏ qua phần đã làm xong).

| Bước | File | Kết quả |
|---|---|---|
| 1 | `s01_prepare_data.py` | `data/images/`, `data/split.csv` |
| 1b | `s02_sift_probe.py` | `out/sift_probe.csv`, `out/sift_probe.png` |
| 2 | `s03_sift_extract.py` | `data/sift.npz`, `out/sift_stats.csv` |
| 3 | `s04_codebook.py` | `data/codebook_k{100,300}.npy`, `data/words.npz`, `out/codebook_stats.csv` |
| 4 | `s05_transactions.py` | `data/transactions/k{100,300}_word/`, `out/transaction_stats.csv`, `out/tau_sensitivity.csv` |
| 5 | `s06_mine.py` | `data/patterns/`, `out/mining_sweep.csv`, `out/minsup_choice.csv`, `out/fpclose_check.csv` |
| 6–7 | `s07`, `s08` | *(đang làm)* |

## Thư viện sử dụng

Theo hướng dẫn của thầy, phần tiền xử lý được dùng thư viện hỗ trợ. **Khai thác tập phổ biến chỉ dùng FPClose do nhóm tự cài đặt (Julia)**; không dùng mlxtend, efficient-apriori hay thư viện FIM nào khác. SPMF chỉ dùng để đối chiếu kết quả.

**Tiền xử lý (biến ảnh thành giao dịch)**

| Thư viện | Dùng để làm gì | Bước |
|---|---|---|
| OpenCV (`opencv-python-headless`) | Đọc ảnh, chuyển xám, resize; trích đặc trưng SIFT | 1, 2 |
| NumPy | Mảng descriptor, tọa độ keypoint, histogram | 2–4 |
| scikit-learn | Chia train/test theo lớp (`train_test_split`); gom cụm tạo codebook (`MiniBatchKMeans`); đo accuracy CV khi chọn τ (`StratifiedKFold`, `cross_val_score`, `LogisticRegression`) | 1, 3, 4 |
| pandas | Danh sách ảnh, bảng thống kê | 1–4 |

**Khai thác:** FPClose của nhóm (`demo.jl`), bước 5.

**Phân tích và đánh giá sau khai thác**

| Thư viện | Dùng để làm gì | Bước |
|---|---|---|
| pandas | Bảng mẫu, growth rate, bảng kết quả | 6, 7 |
| Matplotlib | Vẽ keypoint và mẫu lên ảnh, biểu đồ | 1b, 6, 7 |
| scikit-learn | Phân loại (`LogisticRegression`), accuracy, macro-F1, confusion matrix | 7 |


## 0. Thiết lập môi trường

In [ ]:
import os, sys, subprocess
from pathlib import Path

USE_OWN_IMPL = True      # True: khai thác bằng FPClose (Julia) của nhóm. Bản nộp PHẢI là True.
REPO_URL = "https://github.com/DRAGON-1312/FPClose.git"
REPO_BRANCH = "main"     # muốn thử trên Colab trước khi merge: đổi thành nhánh đang làm, vd "feature/app2"
APP_REL = Path("applications/app2_image")
IN_COLAB = "google.colab" in sys.modules

def find_repo_dir():
    """Gốc repo FPClose: đi từ thư mục hiện tại lên trên, tìm thư mục có applications/app2_image."""
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / APP_REL / "src" / "s01_prepare_data.py").exists():
            return d
    return None

REPO_DIR = find_repo_dir()
if REPO_DIR is None and IN_COLAB:                  # Colab: clone repo về /content/FPClose (một lần)
    REPO_DIR = Path("/content/FPClose")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-q", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
assert REPO_DIR is not None and (REPO_DIR / APP_REL / "src").is_dir(), \
    "Không tìm thấy applications/app2_image: mở notebook từ trong repo FPClose, hoặc chạy trên Colab"
APP_DIR = REPO_DIR / APP_REL

os.chdir(APP_DIR)                                  # các bước dùng đường dẫn tương đối src/, data/, out/
os.environ["PYTHONIOENCODING"] = "utf-8"           # để in tiếng Việt trên console Windows
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if str(APP_DIR / "src") not in sys.path:
    sys.path.insert(0, str(APP_DIR / "src"))

def run_step(script, *args):
    """Chạy một bước trong src/ bằng đúng Python của kernel, in output, dừng nếu lỗi."""
    r = subprocess.run([sys.executable, f"src/{script}", *map(str, args)],
                       capture_output=True, text=True, encoding="utf-8")
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError(f"{script} lỗi (mã {r.returncode})")

import config
print("REPO_DIR =", REPO_DIR, "| Colab:", IN_COLAB, "| SEED =", config.SEED)

## 1. Chuẩn bị dữ liệu

- **5 lớp** `airplanes`, `Motorbikes`, `Faces_easy`, `watch`, `Leopards`: hình dạng khác nhau rõ, đủ ảnh. Bỏ `Faces` (trùng `Faces_easy`) và `BACKGROUND_Google` (không phải một loại vật thể).
- **200 ảnh/lớp**: `Leopards` chỉ có 200 ảnh, nên đây là mức cân bằng lớn nhất. Lớp cân bằng thì support giữa các lớp mới so sánh được (growth rate).
- **Train/test 70/30 theo từng lớp** (140/60), seed 42. Mẫu chỉ khai thác trên train.
- **Ảnh xám, cạnh dài 400 px**: SIFT chỉ dùng độ sáng; cùng kích thước để số keypoint không phụ thuộc độ phân giải gốc.

In [ ]:
run_step("s01_prepare_data.py")

In [ ]:
import pandas as pd
split = pd.read_csv(config.DATA_SPLIT)
pd.crosstab(split.label, split.split, margins=True)

## 1b. Chạy thử SIFT

`cv2.SIFT_create(nfeatures=500)` trên 10 ảnh train mỗi lớp, để xem số keypoint trung bình trước khi chạy toàn bộ.

In [ ]:
run_step("s02_sift_probe.py", 10)

In [ ]:
from IPython.display import Image, display
probe = pd.read_csv(config.OUT / "sift_probe.csv")
display(probe.groupby("label").n_kp.agg(["count", "mean", "min", "max"]).round(1))
Image(str(config.OUT / "sift_probe.png"))

**Nhận xét (lần chạy thử):**

- Trung bình ~437 keypoint/ảnh, descriptor 128 chiều → 700 ảnh train cho ~300k descriptor, lấy mẫu 150k cho k-means là đủ.
- `airplanes` ít keypoint hơn hẳn (~230) vì nền trời trơn → giao dịch ngắn hơn, cần lưu ý khi chọn minsup theo lớp.
- Hầu hết các lớp chạm trần 500 → giữ `nfeatures=500` để độ dài giao dịch giữa các lớp tương đương.
- **Thiên lệch nền:** `Faces_easy` có nhiều keypoint trên kệ sách phía sau, `Leopards` có cỏ và viền đen. Sẽ phân tích ở bước vẽ mẫu.

## 2. Trích SIFT toàn bộ ảnh

Chạy `cv2.SIFT_create(nfeatures=500)` trên cả 1000 ảnh (train và test), lưu **descriptor** (128 chiều) và **tọa độ keypoint** (x, y, size, angle) vào `data/sift.npz`.

- **Vì sao lưu cả test:** ở bước 3 mọi keypoint (train và test) đều được gán từ thị giác bằng codebook; codebook chỉ học từ train nên không rò rỉ.
- **Vì sao lưu tọa độ:** bước vẽ mẫu lên ảnh gốc và mã hóa vị trí lưới 2×2 cần biết keypoint nằm ở đâu.
- **Cách lưu:** một mảng phẳng + `offsets` (keypoint của ảnh *i* nằm ở `[offsets[i], offsets[i+1])`), thứ tự ảnh = thứ tự dòng của `data/split.csv`. Descriptor của OpenCV vốn là số nguyên 0–255 nên lưu `uint8` không mất thông tin (nhỏ hơn 4 lần, file ~49 MB).
- SIFT là thuật toán xác định, không có bước ngẫu nhiên nên chạy lại cho kết quả giống hệt.

In [ ]:
run_step("s03_sift_extract.py")

In [ ]:
import numpy as np
sift_stats = pd.read_csv(config.OUT / "sift_stats.csv")
z = np.load(config.DATA_SIFT)
is_train = (sift_stats.split == "train").to_numpy()
n_kp = np.diff(z["offsets"])
print(f"Descriptor: {z['desc'].shape}, train {n_kp[is_train].sum():,} / test {n_kp[~is_train].sum():,} keypoint")
sift_stats.pivot_table(index="label", columns="split", values="n_kp", aggfunc="mean").round(1).loc[config.CLASSES]

**Nhận xét:**

- 1000 ảnh cho ~450k keypoint, trong đó **~316k descriptor thuộc train** → bước 3 lấy mẫu khoảng 150k descriptor (một nửa) cho k-means là hợp lý, đủ phủ mà không tốn thời gian.
- Khác lần chạy thử 10 ảnh (airplanes ~230), toàn bộ train cho airplanes trung bình ~282 keypoint/ảnh; vẫn thấp hơn hẳn 4 lớp còn lại (~480–500, gần chạm trần 500). Giao dịch của airplanes sẽ **ngắn hơn** → cùng một minsup phần trăm nhưng mật độ khác, cần xem ở bảng thống kê bước 4.
- Không ảnh nào thiếu keypoint (ít nhất 75 keypoint), nên không phải loại ảnh nào.

## 3. Codebook và gán từ thị giác

Gom các descriptor SIFT thành *k* cụm bằng `MiniBatchKMeans`; tâm mỗi cụm là một **từ thị giác** (một kiểu mảnh ảnh cục bộ). Mỗi keypoint được gán từ của tâm gần nhất. Một ảnh khi đó là một "túi từ", và là một giao dịch ở bước sau.

- **Chỉ học từ train.** Mẫu lấy từ 700 ảnh train; test chỉ được *gán từ* bằng codebook đã học, không tham gia học. Nếu test lọt vào k-means thì thông tin test rò sang bước phân loại.
- **~150k descriptor, đều theo ảnh:** mỗi ảnh train đóng góp tối đa *q* descriptor (*q* tìm tự động, ra 221), ảnh ít hơn *q* thì lấy hết. Lấy ngẫu nhiên thuần túy sẽ để Motorbikes/Leopards (~500 keypoint/ảnh) chiếm phần lớn, còn airplanes (~282) bị lép, nên codebook sẽ thiên về các lớp nhiều keypoint. 150k là gần một nửa số descriptor train, đủ phủ mà k-means vẫn chạy trong vài giây.
- **k = 100 và 300**, so sánh cả hai ở phần phân loại:
  - k nhỏ: mỗi từ thô, xuất hiện trong nhiều ảnh → support cao, giao dịch "dày", FPClose cho ra nhiều mẫu nhưng kém đặc trưng.
  - k lớn: từ chi tiết hơn, phân biệt lớp tốt hơn nhưng support thấp hơn → phải hạ minsup, giao dịch thưa.
- **`MiniBatchKMeans`** thay vì `KMeans`: nhanh hơn nhiều với dữ liệu lớn. Đã kiểm tra: inertia chỉ cao hơn `KMeans` đầy đủ ~1,8% (k=100 và k=300) nên chất lượng gần như tương đương. `n_init=3`, `batch_size=4096`, `random_state=42`; chạy lại cho codebook giống hệt (đã so hash file).
- Từ được đánh số 0…k−1 ở bước này; bước giao dịch mới đổi sang item SPMF bắt đầu từ 1.

In [ ]:
run_step("s04_codebook.py")

In [ ]:
words = np.load(config.DATA_WORDS)
print({name: (w.shape, w.dtype) for name, w in words.items()}, "| khớp số keypoint của sift.npz:", all(len(w) == len(z["x"]) for w in words.values()))
pd.read_csv(config.OUT / "codebook_stats.csv")

**Nhận xét:**

- Không từ nào rỗng trên train (`unused_words_train = 0`). Cụm lớn nhất gấp ~3–5 lần cụm trung vị (k=100: 9845 so với 2891; k=300: 4522 so với 935), tức vài từ rất phổ biến; nhiều khả năng là vùng phẳng/nền. Những từ phổ biến ở cả 5 lớp sẽ bị bước **stop word thị giác** (>80% ảnh train) loại ở bước giao dịch.
- Cụm nhỏ nhất vẫn có 169 keypoint (k=300) nên mọi từ đều có dữ liệu để thành item.
- Inertia giảm từ k=100 sang k=300 (1,18e10 → 1,04e10) vì nhiều cụm hơn, không so sánh trực tiếp để chọn k: k được chọn qua kết quả phân loại.

## 4. Giao dịch (transactions)

Mỗi ảnh thành một giao dịch: tập các từ thị giác "đủ bền" của ảnh đó. Item SPMF = từ + 1 (SPMF cần số dương); id giữ ổn định, không đánh số lại sau khi lọc.

- **Ngưỡng τ = 3:** chỉ giữ từ xuất hiện ít nhất 3 lần trong ảnh. Một từ chỉ xuất hiện một lần thường là nhiễu lượng tử hóa của k-means; xuất hiện lặp lại mới cho thấy một cấu trúc hình ảnh thật. Giá trị mặc định ban đầu là τ = 2; τ = 3 được chọn bằng **tiêu chí đặt trước** (xem bên dưới).
- **Stop word thị giác:** bỏ item có mặt trong **hơn 80% ảnh train** (chỉ đếm trên train, rồi áp cho cả test). Độ phổ biến tính *sau* khi lọc τ, vì đó mới là support mà FPClose nhìn thấy. Với 5 lớp cân bằng, vượt 80% nghĩa là có mặt ở cả 5 lớp nên không phân biệt được lớp nào.
- **Chỉ chế độ `word` được ghi file và khai thác.** Đề bài định nghĩa giao dịch là tập visual words. Chế độ `grid` (item = (từ, ô của lưới 2×2), `item = từ*4 + ô + 1`) chỉ có trong **bảng nhạy cảm**, để xem thông tin vị trí giúp phân loại đến đâu; nó không đi tiếp sang khai thác. Đây là quyết định phạm vi (đề không yêu cầu vị trí, thời gian có hạn), không dựa trên kết quả. Lưu ý: ở `grid`, τ áp lên cặp (từ, ô) chứ không phải từ, nên τ của hai chế độ không so trực tiếp được.
- **Ảnh rỗng** (sau lọc không còn item nào) bị **loại khỏi file .txt**, vì `read_spmf` của FPClose bỏ dòng trống nên số dòng sẽ lệch so với thứ tự ảnh. Ảnh đó được đánh dấu `empty=True` trong `transactions_index.csv` (`line_common`, `line_class` để trống). Số giao dịch *n* trong `ceil(minsup × n)` là số dòng không rỗng của file. Ở cấu hình đã chọn không có ảnh rỗng nào.
- **File xuất** (`data/transactions/k{100,300}_word/`): mỗi lớp một file (chỉ train), `all_train.txt` (file chung = toàn bộ train, cần để tính support ở các lớp khác cho growth rate), `test.txt` (chỉ để kiểm tra ảnh test có chứa mẫu hay không), `items.csv` (item ↔ từ, số ảnh chứa, có là stop word không).

### Chọn τ: tiêu chí đặt *trước* khi xem kết quả

Tiêu chí được viết vào docstring của `s05_transactions.py` và commit (`a541b25`) **trước khi chạy script**:

> Với từng (k, chế độ), cố định stop word 80%. Gọi A* là accuracy CV 5-fold (trên train) cao nhất trong τ ∈ {1, 2, 3}. Chọn **τ lớn nhất** có accuracy ≥ A* − 0,01.

Lý do: τ lớn hơn cho giao dịch ngắn và thưa hơn, FPClose chạy nhanh hơn và bớt nhiễu; chênh lệch dưới 1 điểm % không đủ để ưu tiên τ nhỏ. Accuracy lấy từ `LogisticRegression(random_state=42)` trên vector 0/1 "ảnh có item này hay không", `StratifiedKFold(5, shuffle=True, random_state=42)`, **chỉ trên train**. Giới hạn: ngưỡng stop word tính trên toàn bộ train trước khi chia fold (không dùng nhãn, nhưng không lồng trong fold).

**Áp dụng:** tiêu chí chọn τ = 3 cho cả k = 100 và k = 300 ở chế độ `word`, nên `TAU = 3`. Trước khi đổi đã kiểm tra thêm một điều kiện không dính đến accuracy: với τ = 3 không có ảnh rỗng nào, airplanes vẫn còn trung bình 31–38 item mỗi ảnh.

In [ ]:
run_step("s05_transactions.py")

In [ ]:
tx = pd.read_csv(config.OUT / "transaction_stats.csv")
tx.drop(columns=["tau", "stop"]).set_index(["k", "variant", "group"])

In [ ]:
sens = pd.read_csv(config.OUT / "tau_sensitivity.csv")
display(sens[sens.stop == config.STOP_DF].drop(columns="stop").set_index(["k", "variant", "tau"]))
display(pd.read_csv(config.OUT / "tau_recommendation.csv"))
sens.pivot_table(index=["k", "variant", "tau"], columns="stop", values=["n_stop_items", "cv_acc"])

**Nhận xét (τ = 3, stop word 80%, chế độ `word`):**

- **Không có ảnh rỗng:** 140 giao dịch mỗi lớp, 700 train, 300 test, ở cả k = 100 và k = 300.
- **k = 100 vẫn dày:** độ dài TB 57,0 trên 99 item, mật độ 0,58 (Motorbikes 0,71, airplanes 0,38). Giao dịch gần "đầy" nên FPClose sẽ nặng nhất ở cấu hình này; minsup cần bắt đầu cao. k = 300: độ dài TB 58,1 trên 300 item, mật độ 0,19.
- **airplanes ngắn bằng khoảng một nửa các lớp khác** (k = 100: 37,7 so với 53,8–69,9; k = 300: 31,5 so với 62,8–65,8), hệ quả của ít keypoint (nền trời trơn) và τ = 3 làm khoảng cách này rộng thêm một chút. Cùng một minsup phần trăm, mẫu của airplanes sẽ ngắn hơn và ít hơn; cần nhớ điều này khi so mẫu giữa các lớp.
- **Có ảnh rất ít item:** ngắn nhất 6 item (k = 100) và 2 item (Faces_easy, k = 300). Những ảnh này gần như không thể chứa mẫu dài.
- **Stop word 80% gần như không có tác dụng:** ở τ = 3 chỉ loại 1 từ (k = 100) và 0 từ (k = 300). Bảng nhạy cảm cho thấy nó chỉ loại nhiều khi τ = 1 (84/100 từ ở k = 100). Báo cáo nên mô tả đúng: bước này có hiệu lực chủ yếu ở k nhỏ và τ nhỏ, không phải bước lọc mạnh.
- **Chế độ `grid` trong bảng nhạy cảm:** ở cùng τ, thêm vị trí thường làm accuracy CV cao hơn (k = 100, τ = 3: 0,889 so với 0,796 của `word`; k = 300, τ = 2: 0,900 so với 0,850), trừ k = 300, τ = 3, nơi `grid` thấp hơn (0,831 so với 0,876) vì giao dịch còn quá thưa (~33 item trên 1200). Khi `grid` hơn, nhiều khả năng là vì vật thể trong Caltech-101 thường nằm cùng vị trí và cùng hướng, một thiên lệch của bộ dữ liệu. (Tiêu chí, nếu áp cho `grid`, sẽ chọn τ = 3 ở k = 100 và τ = 1 ở k = 300; không dùng vì `grid` không được khai thác.)
- **Lưu ý khi đọc accuracy:** độ lệch chuẩn giữa các fold khoảng 1–3 điểm %, bằng hoặc hơn nhiều chênh lệch giữa các τ. Accuracy đo trên *mọi item*, không phải trên các mẫu khai thác, nên chỉ là chỉ báo gián tiếp cho việc chọn τ.

## 5. Khai thác tập phổ biến đóng bằng FPClose của nhóm

**Mọi mẫu trong ứng dụng này đều do cài đặt FPClose (Julia) của nhóm khai thác**, không dùng thư viện FIM nào (`USE_OWN_IMPL = True`). Python chỉ gọi `demo.jl` của nhóm qua `subprocess`, mỗi lần một file và một minsup:

```
julia --project=<repo> demo.jl --minsup 55% --output-closed <output> data/transactions/k100_word/class_<lớp>.txt
```

- **Khai thác theo từng lớp, chỉ trên train** (`class_<lớp>.txt`). Ảnh test không tham gia khai thác.
- `--minsup` dạng phần trăm, ngưỡng số đếm = `ceil(minsup × n)` với *n* = số giao dịch của lớp (140), giống SPMF. Output dạng `1 2 3 #SUP: 5`, `#SUP` là số đếm.
- FPClose cho **tập phổ biến đóng** (closed frequent itemset): tập *X* phổ biến mà không có tập cha nào cùng support. Tập đóng giữ đủ thông tin của mọi tập phổ biến (suy ra được tất cả cùng support) nhưng ít hơn nhiều, nên dễ phân tích hơn.

### Chọn minsup: tiêu chí đặt *trước* khi quét

Tiêu chí được viết vào docstring của `s06_mine.py` và commit (`53b9642`) **trước khi chạy quét**:

> Với mỗi k, quét minsup giảm dần (k = 100: 90% → 50%; k = 300: 60% → 10%; bước 5%). Ở mỗi mức chạy FPClose cho cả 5 lớp. Chọn **minsup cao nhất** mà **mọi lớp** đều có **≥ 50 tập đóng dài ≥ 2 item**. Dùng chung một minsup cho 5 lớp của cùng k. Dừng quét ngay khi đạt.

Lý do:
- **minsup cao:** mẫu có mặt ở nhiều ảnh của lớp nên đại diện cho lớp, ít do ngẫu nhiên; growth rate của mẫu support thấp rất dao động.
- **≥ 50 mẫu dài ≥ 2 mỗi lớp:** bước 6 cần top-10 mỗi lớp, bước 7 cần top-N mẫu (≤ 20) mỗi lớp xếp theo growth rate, nên cần nhóm ứng viên lớn hơn vài lần. Dài ≥ 2 vì câu hỏi là về **tổ hợp** chi tiết; mẫu 1 item chỉ là một từ phổ biến.
- **Chung minsup cho 5 lớp:** cùng ngưỡng tương đối nên mẫu giữa các lớp so được; lớp có giao dịch ngắn nhất sẽ quyết định mức minsup.

Công khai: trước khi viết tiêu chí đã chạy thử minsup 60% trên 3 file để chọn lưới quét, chưa xem các mức khác.

**Kiểm tra độc lập bằng Python** (không thay cho unit test của FPClose): đếm lại support của *mọi* tập đóng trên file giao dịch bằng bitset, và kiểm tra tính đóng (thêm bất kỳ item nào cũng làm support giảm) trên tối đa 5000 tập mỗi lớp. Không kiểm tra được tính đầy đủ (cần khai thác lại).

In [ ]:
# Julia cho FPClose: trên Colab cài bản 1.13.4 từ tarball chính thức vào /usr, y như notebooks/demo.ipynb của nhóm.
# Trên máy cá nhân: cài sẵn (Windows: winget install Julialang.Juliaup; juliaup add 1.13.4; juliaup default 1.13.4).
JULIA_VERSION = "1.13.4"   # trùng demo.ipynb và máy đo chính thức của nhóm

def julia_version(exe="julia"):
    try:
        return subprocess.run([exe, "--version"], capture_output=True, text=True).stdout.strip()
    except FileNotFoundError:
        return ""

WANT = f"julia version {JULIA_VERSION}"
if IN_COLAB and julia_version() != WANT:
    if julia_version("/usr/bin/julia") != WANT:
        mm = ".".join(JULIA_VERSION.split(".")[:2])
        url = f"https://julialang-s3.julialang.org/bin/linux/x64/{mm}/julia-{JULIA_VERSION}-linux-x86_64.tar.gz"
        print("Tải", url)
        subprocess.run(f"curl -fsSL -o /tmp/julia.tar.gz {url} && tar -xzf /tmp/julia.tar.gz -C /usr "
                       "--strip-components 1 && rm -f /tmp/julia.tar.gz", shell=True, check=True)
    # Colab có sẵn một bản Julia khác ở /usr/local/bin (đứng trước /usr/bin trong PATH): trỏ lệnh julia về bản vừa cài.
    if julia_version("/usr/bin/julia") == WANT and os.path.isdir("/usr/local/bin"):
        subprocess.run(["ln", "-sf", "/usr/bin/julia", "/usr/local/bin/julia"], check=True)
print("Julia:", julia_version() or "KHÔNG TÌM THẤY")
assert julia_version() == WANT, f"cần {WANT} trên PATH để chạy FPClose của nhóm"


In [ ]:
assert USE_OWN_IMPL, "Bản nộp phải khai thác bằng FPClose của nhóm"
run_step("s06_mine.py")

In [ ]:
sweep = pd.read_csv(config.OUT / "mining_sweep.csv")
for k in config.K_LIST:
    print(f"k = {k}: số tập đóng dài >= 2 (hàng: minsup %, cột: lớp)")
    display(sweep[sweep.k == k].pivot(index="minsup_pct", columns="class", values="n_len2")
            .sort_index(ascending=False)[config.CLASSES])
display(pd.read_csv(config.OUT / "minsup_choice.csv"))
pd.read_csv(config.OUT / "fpclose_check.csv")

**Nhận xét (kết quả lần chạy):**

- **minsup được chọn: k = 100 → 55%** (mẫu phải có ở ≥ 77/140 ảnh), **k = 300 → 25%** (≥ 35/140 ảnh). Ở cả hai, **airplanes là lớp quyết định** (63 và 102 tập dài ≥ 2, các lớp khác vượt xa ngưỡng từ trước). Đúng như dự kiến: airplanes có giao dịch ngắn nhất (TB 37,7 và 31,5 item).
- **Số mẫu chênh nhau hàng nghìn lần giữa các lớp** ở cùng minsup:

  | k (minsup) | airplanes | watch | Motorbikes | Faces_easy | Leopards |
  |---|---|---|---|---|---|
  | 100 (55%) | 77 | 2.477 | 86.852 | 124.779 | 381.587 |
  | 300 (25%) | 132 | 1.145 | 650 | 5.963 | 2.604.564 |

  (tổng số tập đóng, mọi độ dài). **Leopards bùng nổ** dù mật độ giao dịch của nó *không* cao nhất (k = 100: 0,54 so với 0,71 của Motorbikes). Giả thuyết: lông báo là một **kết cấu lặp lại**, nên cùng một nhóm từ thị giác xuất hiện cùng nhau trong gần như mọi ảnh Leopards, sinh ra rất nhiều tổ hợp phổ biến. Sẽ kiểm tra ở bước 6 khi xem các từ đó nằm ở đâu trên ảnh.
- **Số mẫu không đo mức độ đặc trưng.** Nhiều mẫu chỉ có nghĩa là các từ hay đi cùng nhau trong lớp đó; mẫu có đặc trưng cho lớp hay không phải xét bằng growth rate (bước 6).
- **Chi phí:** lần gọi nặng nhất là k = 300, Leopards, 25%: 2,6 triệu tập đóng, dài tới 21 item, khai thác 60 giây. Bảng mẫu k = 300 nặng ~155 MB nên không đưa lên Hugging Face; chạy lại bước này để sinh lại.
- **Kiểm tra độc lập:** support của mọi tập đóng đều đúng, tính đóng đúng trên toàn bộ mẫu kiểm tra (tối đa 5000 tập mỗi lớp), và ngưỡng số đếm của FPClose trùng với `ceil(minsup × n)` tính bằng số nguyên.